# Nifty 100 data downloader
Downloads about 1 year of daily prices (Open, High, Low, Close, Volume) for every Nifty 100 stock and saves them in **one Excel file** (one sheet per stock, plus a `Summary` sheet).
Run the cells top to bottom, then upload `nifty100_prices.xlsx` to Claude.

In [ ]:
!pip -q install yfinance openpyxl pandas requests

## 1. Get the Nifty 100 constituents

In [ ]:
import io, time, requests, pandas as pd

URL = "https://www.niftyindices.com/IndexConstituent/ind_nifty100list.csv"
HEADERS = {"User-Agent": "Mozilla/5.0"}   # niftyindices.com serves the file to browsers

cons = None
try:
    r = requests.get(URL, headers=HEADERS, timeout=30)
    r.raise_for_status()
    cons = pd.read_csv(io.StringIO(r.text))
    print("Downloaded list from niftyindices.com")
except Exception as e:
    print("Could not download the list automatically:", e)
    print("Download ind_nifty100list.csv from niftyindices.com (Indices > Broad Based > NIFTY 100 > Index Constituents) and upload it below.")
    from google.colab import files
    up = files.upload()
    cons = pd.read_csv(io.BytesIO(next(iter(up.values()))))

cons.columns = [c.strip() for c in cons.columns]
symbols = cons["Symbol"].astype(str).str.strip().tolist()
names = dict(zip(symbols, cons.get("Company Name", cons["Symbol"])))
industry = dict(zip(symbols, cons.get("Industry", [""] * len(symbols))))
print(len(symbols), "stocks:", symbols[:8], "...")

## 2. Download daily prices

In [ ]:
import yfinance as yf

PERIOD = "1y"        # change to "2y", "6mo" etc. if you want
frames, failed = {}, []

def fetch(sym, tries=3):
    for i in range(tries):
        try:
            df = yf.Ticker(sym + ".NS").history(period=PERIOD, interval="1d", auto_adjust=False)
            if not df.empty:
                return df
        except Exception:
            pass
        time.sleep(2 * (i + 1))
    return None

for n, sym in enumerate(symbols, 1):
    df = fetch(sym)
    if df is None:
        failed.append(sym); print(f"[{n}/{len(symbols)}] NO DATA  {sym}")
        continue
    df = df[["Open", "High", "Low", "Close", "Volume"]].round(2)
    df.index = pd.to_datetime(df.index).tz_localize(None).strftime("%Y-%m-%d")
    df.index.name = "Date"
    frames[sym] = df
    print(f"[{n}/{len(symbols)}] ok  {sym}  {len(df)} rows")
    time.sleep(0.7)      # be polite to Yahoo

print("\nDone.", len(frames), "downloaded,", len(failed), "failed:", failed)

**If some stocks failed:** names that differ on Yahoo (for example a renamed or newly listed stock) fail here. Fix the ticker in the `retry_map` below and run the next cell.

In [ ]:
retry_map = {
    # "OLDSYMBOL": "YAHOO_TICKER_WITHOUT_.NS",
}
for sym in list(failed):
    alt = retry_map.get(sym)
    if not alt:
        continue
    df = fetch(alt)
    if df is not None:
        df = df[["Open", "High", "Low", "Close", "Volume"]].round(2)
        df.index = pd.to_datetime(df.index).tz_localize(None).strftime("%Y-%m-%d")
        df.index.name = "Date"
        frames[sym] = df
        failed.remove(sym)
        print("fixed", sym)
print("Still failed:", failed)

## 3. Build the Excel file

In [ ]:
import numpy as np

def rsi14(close):
    d = close.diff()
    up = d.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
    dn = (-d.clip(upper=0)).ewm(alpha=1/14, adjust=False).mean()
    return 100 - 100 / (1 + up / dn)

rows = []
for sym, df in frames.items():
    c = df["Close"]
    last = c.iloc[-1]
    rows.append({
        "Symbol": sym, "Company": names.get(sym, ""), "Industry": industry.get(sym, ""),
        "Last date": df.index[-1], "Last close": last,
        "1m %": round((last / c.iloc[-22] - 1) * 100, 1) if len(c) > 22 else None,
        "3m %": round((last / c.iloc[-64] - 1) * 100, 1) if len(c) > 64 else None,
        "1y %": round((last / c.iloc[0] - 1) * 100, 1),
        "SMA20": round(c.tail(20).mean(), 2), "SMA50": round(c.tail(50).mean(), 2),
        "SMA200": round(c.tail(200).mean(), 2) if len(c) >= 200 else None,
        "RSI14": round(rsi14(c).iloc[-1], 1),
        "52w high": df["High"].max(), "52w low": df["Low"].min(),
        "% from 52w high": round((last / df["High"].max() - 1) * 100, 1),
        "Avg volume 20d": int(df["Volume"].tail(20).mean()),
    })
summary = pd.DataFrame(rows)

OUT = "nifty100_prices.xlsx"
with pd.ExcelWriter(OUT, engine="openpyxl") as xw:
    summary.to_excel(xw, sheet_name="Summary", index=False)
    for sym, df in frames.items():
        df.to_excel(xw, sheet_name=sym[:31])          # Excel sheet names max 31 chars
    if failed:
        pd.DataFrame({"Failed symbols": failed}).to_excel(xw, sheet_name="Failed", index=False)

print("Saved", OUT, "with", len(frames), "stock sheets")
summary.head(10)

## 4. Download the file, then upload it to Claude

In [ ]:
from google.colab import files
files.download("nifty100_prices.xlsx")